In [ ]:
import pandas as pd
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/symbolic_data/equations_50k.csv")



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
data = np.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz")

In [ ]:
X_1 = data["X_4"]

In [ ]:
X_1.shape

(500, 1)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import time
import numpy as np

# ==========================================
# 1. CONFIGURATION (The "Tiny" Setup)
# ==========================================
D_MODEL = 24         # Internal dimension
N_HEADS = 4          # 24 / 4 = 6 dim per head
N_LAYERS = 2         # Number of Transformer blocks
FFN_DIM = 48         # Feed-forward expansion
EMB_DIM = 128        # Final JEPA target dimension
MAX_LEN = 32         # Max equation length
BATCH_SIZE = 256
EPOCHS = 30          # Adjust as needed
LR = 1e-3

# Vocabulary Setup
SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>", "<MASK>"]
OPERATORS = ["ADD", "SUB", "MUL", "DIV", "POW", "sin", "cos", "exp", "log"]
VARIABLES = ["x1", "x2", "x3"]
CONSTANTS = [f"<C_{i}>" for i in range(10)]
ALL_TOKENS = SPECIAL_TOKENS + OPERATORS + VARIABLES + CONSTANTS
VOCAB_SIZE = len(ALL_TOKENS)

TOKEN2ID = {t: i for i, t in enumerate(ALL_TOKENS)}
ID2TOKEN = {i: t for i, t in enumerate(ALL_TOKENS)}
PAD_ID, MASK_ID = TOKEN2ID["<PAD>"], TOKEN2ID["<MASK>"]

# ==========================================
# 2. DYNAMIC MASKING DATASET
# ==========================================
class MLMDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Convert string equation to IDs
        tokens = [TOKEN2ID["<BOS>"]] + [TOKEN2ID.get(t, TOKEN2ID["<UNK>"]) for t in self.data[idx].split()] + [TOKEN2ID["<EOS>"]]

        # Padding
        if len(tokens) < MAX_LEN:
            tokens += [PAD_ID] * (MAX_LEN - len(tokens))
        else:
            tokens = tokens[:MAX_LEN]

        tokens = torch.tensor(tokens)
        labels = tokens.clone()

        # Dynamic Masking logic
        # Don't mask Special Tokens (IDs 0-4)
        prob_matrix = torch.full(tokens.shape, 0.15)
        special_tokens_mask = (tokens < 5)
        prob_matrix.masked_fill_(special_tokens_mask, 0.0)

        masked_indices = torch.bernoulli(prob_matrix).bool()

        # Ensure at least one token is masked
        if not masked_indices.any():
            valid_indices = torch.where(~special_tokens_mask)[0]
            masked_indices[valid_indices[torch.randint(0, len(valid_indices), (1,))]] = True

        # Replace with <MASK> ID
        input_ids = tokens.clone()
        input_ids[masked_indices] = MASK_ID

        # Training labels: -100 for non-masked (ignored by CrossEntropy)
        labels[~masked_indices] = -100

        return input_ids, labels, (input_ids != PAD_ID).float()

# ==========================================
# 3. THE 14K PARAMETER MODEL
# ==========================================
class SinusoidalPosEmb(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe_tensor = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe_tensor[:, 0::2] = torch.sin(position * div_term)
        pe_tensor[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe_tensor.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TinyTargetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, D_MODEL, padding_idx=PAD_ID)
        self.pos_enc = SinusoidalPosEmb(D_MODEL, MAX_LEN)

        # Standard Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=FFN_DIM,
            batch_first=True, dropout=0.1, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)

        # Weight Tying: MLM head uses embedding weights
        self.mlm_bias = nn.Parameter(torch.zeros(VOCAB_SIZE))

        # Projection for JEPA (The 128-dim "Thought" vector)
        self.proj = nn.Linear(D_MODEL, EMB_DIM)

    def forward(self, x, mask):
        x = self.embed(x)
        x = self.pos_enc(x)

        # Pass padding mask to transformer (True means "ignore")
        output = self.transformer(x, src_key_padding_mask=(mask == 0))

        # 1. MLM Logits (Weight Tied)
        logits = F.linear(output, self.embed.weight, self.mlm_bias)

        # 2. Mean Pooling for Sequence Embedding
        # mask is (B, T), unsqueeze to (B, T, 1)
        expanded_mask = mask.unsqueeze(-1)
        pooled = (output * expanded_mask).sum(dim=1) / expanded_mask.sum(dim=1).clamp(min=1e-9)
        equation_emb = self.proj(pooled)

        return logits, equation_emb

# ==========================================
# 4. TRAINING & UTILITIES
# ==========================================

# Dummy data generator for Colab testing

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (Replace with your CSV loading logic)
raw_data = data['expression_prefix_masked']
dataset = MLMDataset(raw_data)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = TinyTargetEncoder().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# Param count check
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {params:,}")



model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in loader:
        input_ids, labels, mask = [b.to(device) for b in batch]

        logits, _ = model(input_ids, mask)

        # Loss only on masked tokens
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), labels.view(-1), ignore_index=-100)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f}")

# --- Test Prediction ---
model.eval()
test_str = "MUL x1 <C_0>"
input_ids, labels, mask = dataset[0] # Just an example from dataset
input_ids = input_ids.unsqueeze(0).to(device)
mask = mask.unsqueeze(0).to(device)

with torch.no_grad():
    logits, emb = model(input_ids, mask)
    preds = torch.argmax(logits, dim=-1)

print("\n--- Model Test ---")
print(f"Input IDs: {input_ids[0].cpu().tolist()[:5]}")
print(f"Predictions: {preds[0].cpu().tolist()[:5]}")
print(f"Embedding Shape: {emb.shape}")



Using device: cuda


/tmp/ipython-input-277/4040831512.py:106: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)


Total Trainable Parameters: 13,619
Epoch 1 | Loss: 5.1728
Epoch 2 | Loss: 2.6800
Epoch 3 | Loss: 2.4793
Epoch 4 | Loss: 2.3583
Epoch 5 | Loss: 2.2867
Epoch 6 | Loss: 2.2335
Epoch 7 | Loss: 2.1947
Epoch 8 | Loss: 2.1581
Epoch 9 | Loss: 2.1368
Epoch 10 | Loss: 2.1134
Epoch 11 | Loss: 2.0936
Epoch 12 | Loss: 2.0659
Epoch 13 | Loss: 2.0623
Epoch 14 | Loss: 2.0453
Epoch 15 | Loss: 2.0233
Epoch 16 | Loss: 2.0168
Epoch 17 | Loss: 2.0004
Epoch 18 | Loss: 1.9950
Epoch 19 | Loss: 1.9816
Epoch 20 | Loss: 1.9682
Epoch 21 | Loss: 1.9641
Epoch 22 | Loss: 1.9597
Epoch 23 | Loss: 1.9428
Epoch 24 | Loss: 1.9432
Epoch 25 | Loss: 1.9378
Epoch 26 | Loss: 1.9339
Epoch 27 | Loss: 1.9269
Epoch 28 | Loss: 1.9202
Epoch 29 | Loss: 1.9127
Epoch 30 | Loss: 1.9104

--- Model Test ---
Input IDs: [1, 5, 6, 14, 9]
Predictions: [1, 5, 6, 14, 9]
Embedding Shape: torch.Size([1, 128])


In [ ]:
print(data.files)

['X_0', 'y_0', 'X_1', 'y_1', 'X_2', 'y_2', 'X_3', 'y_3', 'X_4', 'y_4', 'X_5', 'y_5', 'X_6', 'y_6', 'X_7', 'y_7', 'X_8', 'y_8', 'X_9', 'y_9', 'X_10', 'y_10', 'X_11', 'y_11', 'X_12', 'y_12', 'X_13', 'y_13', 'X_14', 'y_14', 'X_15', 'y_15', 'X_16', 'y_16', 'X_17', 'y_17', 'X_18', 'y_18', 'X_19', 'y_19', 'X_20', 'y_20', 'X_21', 'y_21', 'X_22', 'y_22', 'X_23', 'y_23', 'X_24', 'y_24', 'X_25', 'y_25', 'X_26', 'y_26', 'X_27', 'y_27', 'X_28', 'y_28', 'X_29', 'y_29', 'X_30', 'y_30', 'X_31', 'y_31', 'X_32', 'y_32', 'X_33', 'y_33', 'X_34', 'y_34', 'X_35', 'y_35', 'X_36', 'y_36', 'X_37', 'y_37', 'X_38', 'y_38', 'X_39', 'y_39', 'X_40', 'y_40', 'X_41', 'y_41', 'X_42', 'y_42', 'X_43', 'y_43', 'X_44', 'y_44', 'X_45', 'y_45', 'X_46', 'y_46', 'X_47', 'y_47', 'X_48', 'y_48', 'X_49', 'y_49', 'X_50', 'y_50', 'X_51', 'y_51', 'X_52', 'y_52', 'X_53', 'y_53', 'X_54', 'y_54', 'X_55', 'y_55', 'X_56', 'y_56', 'X_57', 'y_57', 'X_58', 'y_58', 'X_59', 'y_59', 'X_60', 'y_60', 'X_61', 'y_61', 'X_62', 'y_62', 'X_63', 'y_

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader

def evaluate_feynman_performance(model, df, column_name='expression_prefix_masked'):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    # 1. Prepare Test Dataset from your DataFrame
    feynman_list = df[column_name].astype(str).tolist()
    test_dataset = MLMDataset(feynman_list) # Uses the same class from previous code
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    total_masked_tokens = 0
    correct_masked_predictions = 0
    total_loss = 0

    print(f"Testing on {len(feynman_list)} unseen Feynman equations...\n")

    with torch.no_grad():
        for batch_idx, (input_ids, labels, mask) in enumerate(test_loader):
            input_ids, labels, mask = input_ids.to(device), labels.to(device), mask.to(device)

            logits, _ = model(input_ids, mask)

            # Calculate Loss
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), labels.view(-1), ignore_index=-100)
            total_loss += loss.item()

            # Calculate Masked Accuracy
            # labels has -100 for non-masked tokens, and the original ID for masked ones
            masked_positions = (labels != -100)
            if masked_positions.any():
                predictions = torch.argmax(logits, dim=-1)

                # Compare predictions to actual labels ONLY at masked indices
                correct = (predictions[masked_positions] == labels[masked_positions]).sum().item()
                correct_masked_predictions += correct
                total_masked_tokens += masked_positions.sum().item()

            # Print a few live examples from the first batch
            if batch_idx == 0:
                print("--- Sample Extractions ---")
                for i in range(min(3, len(input_ids))):
                    orig_ids = labels[i].clone()
                    # Reconstruct original from labels and input_ids
                    for idx, val in enumerate(orig_ids):
                        if val == -100: orig_ids[idx] = input_ids[i][idx]

                    input_tokens = [ID2TOKEN[t.item()] for t in input_ids[i] if t != PAD_ID]
                    pred_tokens = [ID2TOKEN[t.item()] for t in predictions[i][:len(input_tokens)]]
                    actual_tokens = [ID2TOKEN[t.item()] for t in orig_ids if t != PAD_ID]

                    print(f"Original: {' '.join(actual_tokens)}")
                    print(f"Masked In: {' '.join(input_tokens)}")
                    print(f"Predicted: {' '.join(pred_tokens)}")
                    print("-" * 10)

    avg_loss = total_loss / len(test_loader)
    accuracy = (correct_masked_predictions / total_masked_tokens) * 100 if total_masked_tokens > 0 else 0

    print(f"\n[TEST RESULTS]")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Masked Token Accuracy: {accuracy:.2f}%")
    return accuracy

# --- HOW TO RUN ---
feynman_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/symbolic_data/feynman_new.csv")
acc = evaluate_feynman_performance(model, feynman_df)

Testing on 100 unseen Feynman equations...

--- Sample Extractions ---
Original: <BOS> DIV exp DIV POW <UNK> <UNK> <C_0> <C_0> <UNK> MUL <C_0> <UNK> <EOS>
Masked In: <BOS> DIV exp DIV POW <UNK> <UNK> <C_0> <C_0> <UNK> MUL <MASK> <UNK> <EOS>
Predicted: <BOS> DIV exp DIV POW <UNK> <UNK> <C_0> <C_0> <UNK> MUL <C_1> <UNK> <EOS>
----------
Original: <BOS> DIV exp DIV POW <UNK> DIV <UNK> <UNK> <C_0> <C_0> MUL <UNK> MUL <C_0> <UNK> <UNK> <EOS>
Masked In: <BOS> DIV <MASK> DIV POW <UNK> DIV <UNK> <UNK> <C_0> <C_0> MUL <UNK> MUL <C_0> <UNK> <UNK> <EOS>
Predicted: <BOS> DIV MUL DIV POW <UNK> DIV <UNK> <UNK> <C_0> <C_0> MUL <UNK> MUL <C_0> <UNK> <UNK> <C_7>
----------
Original: <BOS> DIV exp DIV POW <UNK> DIV SUB <UNK> <UNK> <UNK> <C_0> <C_0> MUL <UNK> MUL <C_0> <UNK> <UNK> <EOS>
Masked In: <BOS> DIV exp <MASK> <MASK> <UNK> DIV SUB <UNK> <UNK> <UNK> <C_0> <C_0> MUL <UNK> MUL <MASK> <UNK> <UNK> <EOS>
Predicted: <BOS> DIV exp MUL MUL <UNK> DIV SUB <UNK> <UNK> <UNK> <C_0> <C_0> MUL <UNK> MUL <C_1> <U

In [ ]:
import torch
from google.colab import files

# Define the filename
MODEL_PATH = "tiny_target_encoder_feynman_21pct.pt"

# Create a dictionary to save (includes weights + metadata)
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'accuracy': 21.0,
    'vocab': TOKEN2ID,  # Saving the vocab is critical for reloading later!
    'config': {
        'd_model': D_MODEL,
        'n_heads': N_HEADS,
        'n_layers': N_LAYERS,
        'emb_dim': EMB_DIM
    }
}

# Save to Colab disk
torch.save(checkpoint, MODEL_PATH)
print(f"Model saved locally to {MODEL_PATH}")

# Trigger browser download to your PC
files.download(MODEL_PATH)

Model saved locally to tiny_target_encoder_feynman_21pct.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# generating target embeddings of 50k equations

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

@torch.no_grad()
def generate_target_embeddings(model, csv_path, device, batch_size=256):
    """
    Passes all 50k equations through the frozen Phase 1 model.
    Returns: A numpy array of shape (50000, 128)
    """
    model.eval()
    model.to(device)

    # 1. Load the strings
    df = pd.read_csv(csv_path)
    # Ensure this matches the column name in your CSV
    equations = df['expression_prefix_masked'].astype(str).tolist()

    all_embeddings = []

    print(f"Generating embeddings for {len(equations)} equations...")

    for i in tqdm(range(0, len(equations), batch_size)):
        batch_strs = equations[i : i + batch_size]

        # 2. Tokenize and Pad the batch
        batch_ids = []
        for s in batch_strs:
            # Re-using the logic from Phase 1: [BOS] + tokens + [EOS]
            tokens = [TOKEN2ID["<BOS>"]] + [TOKEN2ID.get(t, TOKEN2ID["<UNK>"]) for t in s.split()] + [TOKEN2ID["<EOS>"]]
            # Pad to the MAX_LEN used in Phase 1
            if len(tokens) < MAX_LEN:
                tokens += [PAD_ID] * (MAX_LEN - len(tokens))
            else:
                tokens = tokens[:MAX_LEN]
            batch_ids.append(tokens)

        input_ids = torch.tensor(batch_ids).to(device)
        mask = (input_ids != PAD_ID).float().to(device)

        # 3. Get the 128-dim pooled embedding from the model
        # Note: Your model.forward returns (logits, embedding)
        _, embeddings = model(input_ids, mask)

        all_embeddings.append(embeddings.cpu().numpy())

    # Concatenate all batches into one large matrix
    final_matrix = np.concatenate(all_embeddings, axis=0)

    # 4. Save to disk
    np.save("target_embeddings_50k.npy", final_matrix)
    print(f"\nSuccess! Saved matrix of shape {final_matrix.shape} to target_embeddings_50k.npy")

    return final_matrix

# --- EXECUTION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
target_embs = generate_target_embeddings(model, "/content/drive/MyDrive/Colab Notebooks/symbolic_data/equations_50k.csv", device)

Generating embeddings for 50000 equations...


100%|██████████| 196/196 [00:00<00:00, 218.37it/s]



Success! Saved matrix of shape (50000, 128) to target_embeddings_50k.npy


In [ ]:
check_data = np.load("target_embeddings_50k.npy")
print(f"Mean value: {check_data.mean():.4f}")
print(f"Std dev: {check_data.std():.4f}")

Mean value: 0.0414
Std dev: 0.4879


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ==========================================
# 1. DATASET: Points (X,y) -> Target Vector
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []

    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            # Check if index exists AND has data
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue

    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices

# --- Update your Dataset Class to use this list ---
class JEPADataset(Dataset):
    def __init__(self, npz_path, npy_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)
        all_targets = np.load(npy_path)

        # Pre-allocate numpy arrays in RAM to avoid file I/O during training
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.targets = np.zeros((len(valid_indices), 128), dtype=np.float32)

        for i, real_idx in enumerate(tqdm(valid_indices, desc="Pre-loading")):
            x_pts = full_data[f"X_{real_idx}"]
            y_pts = full_data[f"y_{real_idx}"]

            # Efficient slicing
            n_p = min(x_pts.shape[0], 500)
            n_v = min(x_pts.shape[1], 3)

            self.points[i, :n_p, :n_v] = x_pts[:n_p, :n_v]
            self.points[i, :n_p, 3] = y_pts[:n_p]
            self.targets[i] = all_targets[real_idx]

        # Convert to tensors once
        self.points = torch.from_numpy(self.points)
        self.targets = torch.from_numpy(self.targets)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.points[idx], self.targets[idx]# ==========================================
# 2. ARCHITECTURE: Context Encoder & Predictor
# ==========================================
class ContextEncoder(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=256, latent_dim=128):
        super().__init__()
        # Shared MLP for each of the 500 points
        self.point_processor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(500), # Normalize across the 500 points
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )

    def forward(self, x):
        # x: (Batch, 500, 4)
        x = self.point_processor(x) # (Batch, 500, 128)
        # Permutation Invariant Pooling (Mean)
        context_vector = x.mean(dim=1) # (Batch, 128)
        return context_vector

class Predictor(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.ReLU(),
            nn.Linear(dim * 2, dim) # Maps context to target space
        )

    def forward(self, z):
        return self.net(z)

# ==========================================
# 3. TRAINING EXECUTION
# ==========================================
def train_jepa_phase2(npz_file, npy_file, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Get valid indices
    indices = get_valid_indices(npz_file)

    # 2. Load Dataset into RAM
    dataset = JEPADataset(npz_file, npy_file, indices)

    # 3. INCREASE BATCH SIZE: Saturation for GPU
    # Using 1024 or 2048 to fully utilize the GPU cores
    batch_size = 1024
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

    context_encoder = ContextEncoder().to(device)
    predictor = Predictor().to(device)

    # Use a higher Learning Rate with a larger batch size
    optimizer = torch.optim.AdamW(
        list(context_encoder.parameters()) + list(predictor.parameters()),
        lr=2e-3, weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=5e-3, steps_per_epoch=len(loader), epochs=epochs
    )

    criterion = nn.MSELoss()

    print(f"Phase 2 Training Started on {device} (High Utilization Mode)")
    for epoch in range(epochs):
        context_encoder.train()
        predictor.train()
        total_loss = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
        for points, targets in pbar:
            # Move entire batch to GPU at once
            points, targets = points.to(device, non_blocking=True), targets.to(device, non_blocking=True)

            z_context = context_encoder(points)
            z_predicted = predictor(z_context)

            loss = criterion(z_predicted, targets)

            optimizer.zero_grad(set_to_none=True) # Faster than zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            pbar.set_postfix({"MSE": f"{loss.item():.6f}"})

    torch.save(context_encoder.state_dict(), "context_encoder_optimized.pt")
    print("Optimized Phase 2 Models Saved!")

# --- RUN ---
# --- RUN ---
train_jepa_phase2("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz", "target_embeddings_50k.npy")

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [02:44<00:00, 272.36it/s]


Phase 2 Training Started on cuda (High Utilization Mode)


Epoch 50/50: 100%|██████████| 44/44 [00:07<00:00,  6.13it/s, MSE=0.027346]

Optimized Phase 2 Models Saved!


In [ ]:
# context_encoder_optimized.pt  and tiny_target_encoder_feynman_21pct.pt is simple encoder and
#predictor and is underfitting
# the model is complete
# we can try for zero shot on feynman dataset

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ==========================================
# 1. DATASET: Points (X,y) -> Target Vector
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []

    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            # Check if index exists AND has data
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue

    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices

# --- Update your Dataset Class to use this list ---
class JEPADataset(Dataset):
    def __init__(self, npz_path, npy_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)
        all_targets = np.load(npy_path)

        # Pre-allocate numpy arrays in RAM to avoid file I/O during training
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.targets = np.zeros((len(valid_indices), 128), dtype=np.float32)

        for i, real_idx in enumerate(tqdm(valid_indices, desc="Pre-loading")):
            x_pts = full_data[f"X_{real_idx}"]
            y_pts = full_data[f"y_{real_idx}"]

            # Efficient slicing
            n_p = min(x_pts.shape[0], 500)
            n_v = min(x_pts.shape[1], 3)

            self.points[i, :n_p, :n_v] = x_pts[:n_p, :n_v]
            self.points[i, :n_p, 3] = y_pts[:n_p]
            self.targets[i] = all_targets[real_idx]

        # Convert to tensors once
        self.points = torch.from_numpy(self.points)
        self.targets = torch.from_numpy(self.targets)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.points[idx], self.targets[idx]# ==========================================
# 2. ARCHITECTURE: Context Encoder & Predictor
# ==========================================
class TransformerContextEncoder(nn.Module):
    def __init__(self, input_dim=4, embed_dim=128, nhead=4, num_layers=2):
        super().__init__()
        # 1. Project 4D points (x1,x2,x3,y) to a higher dimensional space
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU()
        )

        # 2. Transformer Encoder Layers
        # This allows points to "talk" to each other to find patterns
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=embed_dim * 2,
            batch_first=True,
            dropout=0.05
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Final Prediction Head (The Predictor)
        self.predictor = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.ReLU(),
            nn.Linear(embed_dim * 2, embed_dim)
        )

    def forward(self, x):
        # x shape: (Batch, 500, 4)

        # Project to embedding space
        x = self.input_proj(x) # (Batch, 500, 128)

        # Pass through Transformer
        # We don't need positional encoding because the order of points doesn't matter
        x = self.transformer(x) # (Batch, 500, 128)

        # Global Pooling (Max + Mean) to capture both average and extreme features
        mean_pool = x.mean(dim=1)
        max_pool, _ = x.max(dim=1)
        pooled = mean_pool + max_pool # (Batch, 128)

        # Predict the Phase 1 target embedding
        return self.predictor(pooled)

class Predictor(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.ReLU(),
            nn.Linear(dim * 2, dim) # Maps context to target space
        )

    def forward(self, z):
        return self.net(z)

# ==========================================
# 3. TRAINING EXECUTION
# ==========================================
def train_jepa_phase2_transformer(npz_file, npy_file, epochs=20):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Setup Data
    indices = get_valid_indices(npz_file)
    # Using the FastJEPADataset (RAM loading) we discussed to keep GPU busy
    dataset = JEPADataset(npz_file, npy_file, indices)
    loader = DataLoader(dataset, batch_size=256, shuffle=True, pin_memory=True)

    # 2. Initialize Model
    model = TransformerContextEncoder().to(device)

    # 3. Optimizer & Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-3, steps_per_epoch=len(loader), epochs=epochs
    )

    print(f"Phase 2 Transformer Training Started with Hybrid Loss...")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        epoch_mse = 0
        epoch_cos = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for points, targets in pbar:
            points, targets = points.to(device, non_blocking=True), targets.to(device, non_blocking=True)

            # --- Forward Pass ---
            prediction = model(points)

            # --- HYBRID LOSS CALCULATION ---
            # MSE ensures coordinate accuracy
            mse_val = F.mse_loss(prediction, targets)
            # Cosine ensures directional/semantic accuracy (Lower is better)
            cos_val = 1 - F.cosine_similarity(prediction, targets).mean()

            # Weighted Total Loss
            loss = (0.2 * mse_val) + cos_val

            # --- Backward Pass ---
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            scheduler.step()

            # --- COLLAPSE MONITOR ---
            # Calculate standard deviation of the batch predictions.
            # If std -> 0, all vectors are identical (bad).
            batch_std = prediction.std(dim=0).mean().item()

            epoch_loss += loss.item()
            epoch_mse += mse_val.item()
            epoch_cos += cos_val.item()

            pbar.set_postfix({
                "Total": f"{loss.item():.4f}",
                "Cos": f"{cos_val.item():.4f}",
                "STD": f"{batch_std:.4f}" # Watch this! Keep it above 0.1
            })

        avg_loss = epoch_loss / len(loader)
        avg_mse = epoch_mse / len(loader)
        avg_cos = epoch_cos / len(loader)

        print(f"End of Epoch {epoch+1} | Avg MSE: {avg_mse:.6f} | Avg Cos: {avg_cos:.6f}")

    torch.save(model.state_dict(), "transformer_context_encoder.pt")
    print("Model Saved Successfully!")

# --- EXECUTION ---
# Ensure you call the correct function name
train_jepa_phase2_transformer(
    "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz",
    "target_embeddings_50k.npy"
)



Scanning for valid data points...
Found 44820 valid equations out of 50000.
Loading data into RAM for maximum speed...


Pre-loading: 100%|██████████| 44820/44820 [02:43<00:00, 274.62it/s]


Phase 2 Transformer Training Started with Hybrid Loss...


Epoch 1/20: 100%|██████████| 176/176 [01:02<00:00,  2.80it/s, Total=0.0421, Cos=0.0382, STD=0.0284]


End of Epoch 1 | Avg MSE: 0.085552 | Avg Cos: 0.133673


...

End of Epoch 18 | Avg MSE: 0.023954 | Avg Cos: 0.047708


Epoch 19/20: 100%|██████████| 176/176 [01:02<00:00,  2.82it/s, Total=0.0582, Cos=0.0529, STD=0.0487]


End of Epoch 19 | Avg MSE: 0.023861 | Avg Cos: 0.047513


Epoch 20/20: 100%|██████████| 176/176 [01:02<00:00,  2.83it/s, Total=0.0547, Cos=0.0497, STD=0.0569]

End of Epoch 20 | Avg MSE: 0.023802 | Avg Cos: 0.047391
Model Saved Successfully!
